# Day 6：LLaMA2 + QLoRA 微调实践 — 完整 QLoRA 实验

> **目标**：在真实的 LLaMA-2 模型上完成一次完整的 QLoRA 微调实验——从 NF4 量化加载模型、注入 LoRA 适配器、SFT 数据准备，到训练循环、Loss 监控、权重保存与合并，最终进行文本生成评估；体验 QLoRA 在单卡上微调大模型的全流程。

---

## 总体流程

```
Part 1: 环境准备与模型加载
  安装依赖 → NF4 量化加载 LLaMA-2 → 显存分析

Part 2: LoRA 配置与注入
  选择 target_modules → LoRA 配置 → 可训练参数统计

Part 3: 数据准备
  Alpaca 格式数据 → Prompt Template → Tokenize → DataLoader

Part 4: QLoRA 训练
  训练循环 → Loss 曲线 → 检查点保存

Part 5: 评估与推理
  加载 LoRA 权重 → 文本生成 → 效果评估 → 权重合并
```

**硬件要求**：
- LLaMA-2-7B QLoRA: 单卡 ~10 GB (RTX 3090 / 4090 / A100)
- 如果没有足够显存，可以使用更小的模型或 Day 3 的手写缩小版 LLaMA

In [ ]:
# Part 1: 环境准备
# 确保安装了必要的依赖
# pip install torch transformers peft bitsandbytes accelerate datasets trl

import os
import json
import time
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Optional, List, Dict

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## Part 1：NF4 量化加载 LLaMA-2

使用 bitsandbytes + HuggingFace Transformers 以 NF4 精度加载模型。

**显存对比**：
- FP16 加载 LLaMA-2-7B: ~13 GB
- NF4 加载 LLaMA-2-7B: ~3.5 GB

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

MODEL_NAME = "meta-llama/Llama-2-7b-hf"
# 如果无法访问 LLaMA-2，可以使用其他开源模型:
# MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 更小的模型
# MODEL_NAME = "Qwen/Qwen2-1.5B"  # 另一个选择

# ===== QLoRA 量化配置 =====
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # 4-bit 量化加载
    bnb_4bit_quant_type="nf4",             # NF4 量化类型
    bnb_4bit_use_double_quant=True,        # 双重量化
    bnb_4bit_compute_dtype=torch.bfloat16, # 计算时使用 BF16
)

print("正在加载模型 (NF4 量化)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\n模型加载完成!")
print(f"模型类型: {type(model).__name__}")
print(f"词表大小: {tokenizer.vocab_size}")

if torch.cuda.is_available():
    gpu_mem = torch.cuda.memory_allocated() / 1024**3
    print(f"GPU 显存占用: {gpu_mem:.2f} GB")

## Part 2：LoRA 配置与注入

### 2.1 准备模型用于 QLoRA 训练

`prepare_model_for_kbit_training` 做了几件关键的事：
1. 冻结所有量化层
2. 开启 gradient checkpointing（节省激活值显存）
3. 将 LayerNorm 层转为 FP32（数值稳定性）

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)

# 准备量化模型用于 LoRA 训练
model = prepare_model_for_kbit_training(model)

# ===== LoRA 配置 =====
lora_config = LoraConfig(
    r=16,                          # LoRA 秩
    lora_alpha=32,                 # 缩放因子 α (通常 = 2r)
    target_modules=[               # 目标模块
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",       # FFN (SwiGLU)
    ],
    lora_dropout=0.05,             # Dropout
    bias="none",                   # 不训练 bias
    task_type=TaskType.CAUSAL_LM,  # 因果语言模型
)

print("LoRA 配置:")
print(f"  r = {lora_config.r}")
print(f"  alpha = {lora_config.lora_alpha}")
print(f"  scaling = alpha/r = {lora_config.lora_alpha/lora_config.r}")
print(f"  target_modules = {lora_config.target_modules}")
print(f"  dropout = {lora_config.lora_dropout}")

# 注入 LoRA
model = get_peft_model(model, lora_config)

print("\n" + "="*60)
model.print_trainable_parameters()

if torch.cuda.is_available():
    gpu_mem = torch.cuda.memory_allocated() / 1024**3
    print(f"\nLoRA 注入后 GPU 显存占用: {gpu_mem:.2f} GB")

## Part 3：SFT 数据准备

### 3.1 构造 Alpaca 格式训练数据

使用 Alpaca 风格的 (instruction, input, output) 三元组。

In [ ]:
# Alpaca Prompt Template
PROMPT_WITH_INPUT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

PROMPT_WITHOUT_INPUT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}"""


def format_alpaca(sample: dict) -> str:
    """将 Alpaca 数据格式化为完整 prompt。"""
    if sample.get("input", "").strip():
        return PROMPT_WITH_INPUT.format(**sample)
    return PROMPT_WITHOUT_INPUT.format(**sample)


# 示例数据（实际使用时请加载 Stanford Alpaca 或自定义数据集）
TRAIN_DATA = [
    {"instruction": "What is machine learning?", "input": "",
     "output": "Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed."},
    {"instruction": "Explain the concept in simple terms.",
     "input": "Neural networks are computational systems inspired by biological neural networks.",
     "output": "A neural network is like a digital brain made of connected units that learn to recognize patterns from examples."},
    {"instruction": "What is the capital of France?", "input": "",
     "output": "The capital of France is Paris."},
    {"instruction": "Translate the following to Spanish.", "input": "Good morning, how are you?",
     "output": "Buenos dias, como estas?"},
    {"instruction": "What is LoRA?", "input": "",
     "output": "LoRA stands for Low-Rank Adaptation. It is a parameter-efficient fine-tuning method that freezes the pretrained model weights and injects trainable low-rank decomposition matrices."},
    {"instruction": "Summarize the key idea.",
     "input": "QLoRA uses NF4 quantization to reduce the memory footprint of the base model while adding LoRA adapters for fine-tuning.",
     "output": "QLoRA combines 4-bit quantization with LoRA to enable fine-tuning large models on a single GPU."},
    {"instruction": "Write a haiku about coding.", "input": "",
     "output": "Lines of logic flow\nBugs hide in the tangled code\nDebug finds the truth"},
    {"instruction": "What is gradient descent?", "input": "",
     "output": "Gradient descent is an optimization algorithm that iteratively adjusts parameters in the direction of steepest decrease of a loss function."},
]

# 格式化并查看样本
formatted = format_alpaca(TRAIN_DATA[0])
print("=== 格式化后的样本 ===")
print(formatted)
print(f"\n--- Token 数: {len(tokenizer.encode(formatted))} ---")

In [ ]:
from torch.utils.data import Dataset, DataLoader


class AlpacaSFTDataset(Dataset):
    """Alpaca 格式的 SFT 数据集，带 Loss Mask。"""

    def __init__(self, data, tokenizer, max_len=512):
        self.samples = []
        for item in data:
            # 构造 prompt 部分（不含 output）
            if item.get("input", "").strip():
                prompt = PROMPT_WITH_INPUT.format(
                    instruction=item["instruction"],
                    input=item["input"],
                    output="",
                )
            else:
                prompt = PROMPT_WITHOUT_INPUT.format(
                    instruction=item["instruction"],
                    output="",
                )

            full_text = prompt + item["output"] + tokenizer.eos_token

            encodings = tokenizer(
                full_text,
                truncation=True,
                max_length=max_len,
                padding="max_length",
                return_tensors="pt",
            )

            input_ids = encodings["input_ids"].squeeze()
            attention_mask = encodings["attention_mask"].squeeze()

            # Loss Mask: 只在 output 部分计算 loss
            prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
            prompt_len = len(prompt_ids)

            labels = input_ids.clone()
            labels[:prompt_len] = -100
            labels[attention_mask == 0] = -100

            self.samples.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


train_dataset = AlpacaSFTDataset(TRAIN_DATA, tokenizer, max_len=256)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

print(f"训练数据集大小: {len(train_dataset)}")
sample = train_dataset[0]
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"labels 中非 -100 的 token 数: {(sample['labels'] != -100).sum().item()}")

## Part 4：QLoRA 训练

### 4.1 手写训练循环

虽然可以使用 HuggingFace Trainer 或 TRL 的 SFTTrainer，
但为了学习目的，我们先手写训练循环。

In [ ]:
import bitsandbytes as bnb

# 使用分页 AdamW 优化器
optimizer = bnb.optim.PagedAdamW8bit(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4,
    weight_decay=0.01,
)

# 学习率调度器
from transformers import get_linear_schedule_with_warmup

NUM_EPOCHS = 3
TOTAL_STEPS = NUM_EPOCHS * len(train_loader)
WARMUP_STEPS = int(0.1 * TOTAL_STEPS)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS,
)

print(f"训练配置:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Total steps: {TOTAL_STEPS}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print(f"  Learning rate: 2e-4")
print(f"  Optimizer: PagedAdamW8bit")

In [ ]:
# ===== 训练循环 =====
model.train()
training_losses = []
global_step = 0

print("开始 QLoRA 训练...")
print("=" * 60)

t0 = time.time()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    for step, batch in enumerate(train_loader):
        # 将数据移到设备上
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["labels"].to(model.device)

        # 前向传播
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss

        # 反向传播
        optimizer.zero_grad()
        loss.backward()

        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()),
            max_norm=1.0,
        )

        # 优化器更新
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        training_losses.append(loss.item())
        global_step += 1

    avg_loss = epoch_loss / len(train_loader)
    current_lr = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"LR: {current_lr:.2e} | "
          f"Time: {elapsed:.0f}s")

    if torch.cuda.is_available():
        gpu_mem = torch.cuda.max_memory_allocated() / 1024**3
        print(f"  Peak GPU Memory: {gpu_mem:.2f} GB")

total_time = time.time() - t0
print(f"\n训练完成! 总耗时: {total_time:.0f}s")

### 4.2 使用 HuggingFace Trainer (替代方案)

如果不想手写训练循环，可以使用 TRL 的 SFTTrainer：

In [ ]:
# ===== 使用 SFTTrainer 的替代方案 =====
# from trl import SFTTrainer, SFTConfig
#
# training_args = SFTConfig(
#     output_dir="./qlora_output",
#     num_train_epochs=3,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     weight_decay=0.01,
#     warmup_ratio=0.1,
#     lr_scheduler_type="linear",
#     max_grad_norm=1.0,
#     fp16=False,
#     bf16=True,
#     logging_steps=10,
#     save_strategy="epoch",
#     optim="paged_adamw_8bit",
#     max_seq_length=512,
# )
#
# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     tokenizer=tokenizer,
# )
#
# trainer.train()

print("(SFTTrainer 代码已注释，取消注释即可使用)")

## Part 5：评估与推理

### 5.1 保存 LoRA 权重

In [ ]:
# 保存 LoRA 权重
LORA_OUTPUT_DIR = "./qlora_lora_weights"
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# 查看保存的文件
import os
print("保存的文件:")
for f in os.listdir(LORA_OUTPUT_DIR):
    size = os.path.getsize(os.path.join(LORA_OUTPUT_DIR, f))
    print(f"  {f}: {size/1024/1024:.2f} MB")

### 5.2 文本生成评估

In [ ]:
@torch.no_grad()
def generate_response(model, tokenizer, instruction, input_text="", max_new_tokens=128):
    """使用 Alpaca 格式生成回复。"""
    if input_text.strip():
        prompt = PROMPT_WITH_INPUT.format(
            instruction=instruction, input=input_text, output=""
        )
    else:
        prompt = PROMPT_WITHOUT_INPUT.format(
            instruction=instruction, output=""
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    model.eval()
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
    )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()


# 测试生成
test_prompts = [
    {"instruction": "What is LoRA and why is it useful?", "input": ""},
    {"instruction": "Explain the concept in simple terms.",
     "input": "Quantization reduces the precision of model weights from 32-bit to lower bit representations."},
    {"instruction": "Write a short poem about artificial intelligence.", "input": ""},
]

print("=== QLoRA 微调后的生成效果 ===")
print("=" * 60)
for item in test_prompts:
    print(f"\n[Instruction]: {item['instruction']}")
    if item['input']:
        print(f"[Input]: {item['input']}")
    response = generate_response(model, tokenizer, item['instruction'], item['input'])
    print(f"[Response]: {response}")
    print("-" * 60)

### 5.3 加载已保存的 LoRA 权重

In [ ]:
# 演示如何从保存的 LoRA 权重重新加载
# from peft import PeftModel
#
# # 重新加载基座模型 (NF4)
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
#
# # 加载 LoRA 权重
# model_with_lora = PeftModel.from_pretrained(
#     base_model,
#     LORA_OUTPUT_DIR,
# )
#
# # 现在可以用 model_with_lora 进行推理

print("LoRA 权重加载演示代码已注释")
print("实际使用时取消注释即可")

### 5.4 权重合并（可选）

将 LoRA 权重合并到基座模型中，得到一个标准模型。

**注意**：QLoRA 的合并需要先将 NF4 反量化为 FP16。

In [ ]:
# ===== 权重合并 =====
# 注意: 合并需要先反量化，会占用更多显存

# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("./qlora_merged_model")
# tokenizer.save_pretrained("./qlora_merged_model")
#
# print("合并后的模型已保存")
# print("合并后可以用 GPTQ/AWQ 进一步量化用于推理")

print("权重合并演示代码已注释")
print("\n典型的 QLoRA 部署路径:")
print("  1. QLoRA 训练 → 保存 LoRA 权重 (几十 MB)")
print("  2. 部署时: NF4 基座 + LoRA 权重")
print("  或")
print("  2. 合并: NF4 反量化 → FP16 + LoRA → 合并 → FP16 模型")
print("  3. 再量化: FP16 → GPTQ/AWQ 4-bit → 部署")

## 总结

本日完成了一次完整的 QLoRA 微调实验：

| 步骤 | 内容 |
|------|------|
| 1. 模型加载 | NF4 量化加载 LLaMA-2-7B (~3.5 GB 显存) |
| 2. LoRA 注入 | 7 个 target modules, r=16, α=32 |
| 3. 数据准备 | Alpaca 格式 + Loss Mask |
| 4. 训练 | PagedAdamW8bit + Linear LR Schedule |
| 5. 推理 | 文本生成 + 效果评估 |
| 6. 权重管理 | 保存/加载/合并 LoRA 权重 |

**核心收获**：
- QLoRA 让单卡微调大模型成为现实
- NF4 + 双重量化将 7B 模型压缩到 ~3.5 GB
- LoRA 只增加 ~0.3% 的可训练参数
- 分页优化器避免了训练过程中的 OOM

**明日预告**：Day 7 将深入 LoRA 超参数工程——如何选择最优的 rank、alpha、target_modules，并完成全周知识复盘。